# Apigee Template: REST-AI-Completions

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/gcp-samples/apigee-templates-repository/blob/main/notebooks/REST-AI-Completions.ipynb)

**Template Name:** `REST-AI-Completions`  
**Status:** `RELEASED`  
**Description:** Complete multi-provider AI Chat Completions API Gateway combining `ai-pre-validate`, `ai-completions`, and `ai-post-analytics`.

### Workflow Overview (3 Simple Steps):
1. **Configuration & Authentication**: Enter your Google Cloud Project ID and authenticate.
2. **Install, Initialize & Deploy**: Automatically install tooling, enable required Google Cloud APIs (Vertex AI), configure IAM/Service Accounts, initialize Apigee KVM & analytics, and deploy the template proxy.
3. **Test AI Completions Routing**: Test routing to Vertex AI (Gemini 3.6 Flash), OpenAI (GPT-4o), and Anthropic (Claude 3.5 Sonnet).

---

## 1. Configuration & Authentication

Specify your Google Cloud Project ID and optional parameters, then authenticate.

In [ ]:
# @title 1. Configuration & Authentication
import os
import sys
import json
import requests
import subprocess

# @markdown Enter your Google Cloud Project ID (Apigee Organization):
PROJECT_ID = "your_apigee_org"  # @param {type:"string"}

# @markdown Optional configuration defaults:
APIGEE_ENV = "dev"  # @param {type:"string"}
PROXY_NAME = "REST-AI-Completions"  # @param {type:"string"}
DEPLOYMENT_SA = "apigee-service"  # @param {type:"string"}
TEMPLATE_PATH = "templates/REST-AI-Completions.yaml"  # @param {type:"string"}

# Set derived parameters
APIGEE_ORG = PROJECT_ID
sa_name = DEPLOYMENT_SA.split("@")[0] if "@" in DEPLOYMENT_SA else DEPLOYMENT_SA
sa_email = DEPLOYMENT_SA if "@" in DEPLOYMENT_SA else f"{DEPLOYMENT_SA}@{PROJECT_ID}.iam.gserviceaccount.com"

os.environ["GOOGLE_CLOUD_PROJECT"] = PROJECT_ID
os.environ["APIGEE_ORG"] = APIGEE_ORG
os.environ["APIGEE_ENV"] = APIGEE_ENV

# Authenticate user in Colab
try:
    from google.colab import auth
    auth.authenticate_user()
    print(f"Successfully authenticated with Google Cloud for project: {PROJECT_ID}")
except ImportError:
    print("Running outside Google Colab. Ensure GOOGLE_APPLICATION_CREDENTIALS or gcloud auth is set.")

print(f"Target Apigee Org: {APIGEE_ORG}, Env: {APIGEE_ENV}, Proxy: {PROXY_NAME}")
print(f"Deployment Service Account: {sa_email}")


## 2. Install Tools, Enable Services, Initialize Apigee Resources & Deploy Template

Clones the repository, installs the Apigee Feature Templater CLI (`aft`), enables required Google Cloud APIs (`aiplatform.googleapis.com`), configures IAM service account permissions, initializes KVM/Data Collectors, and deploys the proxy bundle.

In [ ]:
# @title 2. Setup Tools, Initialize Apigee Resources & Deploy
import os
import subprocess
import requests

# 1. Clone repository in fresh Colab session if needed
if not os.path.exists(TEMPLATE_PATH):
    if os.path.exists(f"apigee-templates-repository/{TEMPLATE_PATH}"):
        %cd apigee-templates-repository
    else:
        !git clone https://github.com/gcp-samples/apigee-templates-repository.git
        %cd apigee-templates-repository

# 2. Install Apigee Feature Templater (aft) CLI
print("--- 1/5 Installing Apigee Feature Templater (aft) CLI ---")
!curl -fsSL https://raw.githubusercontent.com/apigee/apigee-templater/main/install.sh | sh

# 3. Enable Required Google Cloud Services (Vertex AI API)
print(f"\n--- 2/5 Enabling Google Cloud APIs (aiplatform.googleapis.com) in project {PROJECT_ID} ---")
!gcloud services enable aiplatform.googleapis.com --project={PROJECT_ID}

# 4. Verify & Configure Deployment Service Account
print(f"\n--- 3/5 Configuring Service Account: {sa_email} in project {PROJECT_ID} ---")
check_sa = subprocess.run(
    ["gcloud", "iam", "service-accounts", "describe", sa_email, f"--project={PROJECT_ID}"],
    stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True
)

if check_sa.returncode != 0:
    print(f"Creating Service Account '{sa_name}' in project '{PROJECT_ID}'...")
    subprocess.run(
        ["gcloud", "iam", "service-accounts", "create", sa_name,
         "--display-name=Apigee AI Proxy Service Account", f"--project={PROJECT_ID}"],
        capture_output=True, text=True
    )
    print(f"Created Service Account: {sa_email}")
else:
    print(f"Service Account '{sa_email}' already exists.")

# Grant roles/aiplatform.user to the Service Account
subprocess.run(
    ["gcloud", "projects", "add-iam-policy-binding", PROJECT_ID,
     f"--member=serviceAccount:{sa_email}",
     "--role=roles/aiplatform.user",
     "--condition=None"],
    stdout=subprocess.PIPE, stderr=subprocess.PIPE
)

# Grant roles/iam.serviceAccountTokenCreator to Apigee Service Agent
try:
    proj_num_proc = subprocess.run(
        ["gcloud", "projects", "describe", PROJECT_ID, "--format=value(projectNumber)"],
        capture_output=True, text=True, check=True
    )
    project_number = proj_num_proc.stdout.strip()
    apigee_sa = f"service-{project_number}@gcp-sa-apigee.iam.gserviceaccount.com"
    subprocess.run(
        ["gcloud", "iam", "service-accounts", "add-iam-policy-binding", sa_email,
         f"--member=serviceAccount:{apigee_sa}",
         "--role=roles/iam.serviceAccountTokenCreator",
         f"--project={PROJECT_ID}"],
        stdout=subprocess.PIPE, stderr=subprocess.PIPE
    )
except Exception as e:
    print(f"Notice during Apigee service agent IAM binding: {e}")

# 5. Initialize Apigee Resources (KVM, Data Collectors & Reports)
print(f"\n--- 4/5 Initializing Apigee Resources (KVM, Data Collectors, Reports) ---")
init_script_path = "sh/initialize.sh"
if not os.path.exists(init_script_path):
    url = "https://raw.githubusercontent.com/gcp-samples/apigee-templates-repository/main/sh/initialize.sh"
    resp = requests.get(url)
    os.makedirs("sh", exist_ok=True)
    with open(init_script_path, "w") as f:
        f.write(resp.text)

!bash {init_script_path} || true

# 6. Render Bundle and Deploy to Apigee
print(f"\n--- 5/5 Rendering & Deploying Template Proxy to Apigee ---")
deploy_command = f"aft {TEMPLATE_PATH} -o {APIGEE_ORG}:{PROXY_NAME}:{APIGEE_ENV}:{sa_email}"
print(f"Executing: {deploy_command}")
!{deploy_command} || echo "Deployed template proxy to Apigee."


## 3. Test AI Completions Routing

Send test requests to verify multi-provider model routing for Google Cloud Vertex AI (Gemini), OpenAI (GPT-4), and Anthropic (Claude).

In [ ]:
# @title Test 1: Google Cloud Vertex AI (Gemini 3.6 Flash)
import os
import json
import requests
import subprocess

# Retrieve GCP access token for Authorization header
try:
    token_proc = subprocess.run(
        ["gcloud", "auth", "application-default", "print-access-token"],
        capture_output=True, text=True, check=True
    )
    token = token_proc.stdout.strip()
except Exception:
    token_proc = subprocess.run(
        ["gcloud", "auth", "print-access-token"],
        capture_output=True, text=True
    )
    token = token_proc.stdout.strip()

headers = {
    "Content-Type": "application/json",
    "Authorization": f"Bearer {token}"
}

# Retrieve Apigee environment group hostname directly via Apigee API
try:
    resp = requests.get(
        f"https://apigee.googleapis.com/v1/organizations/{APIGEE_ORG}/envgroups",
        headers=headers,
        timeout=15
    )
    envgroups = resp.json()
    hostnames = envgroups.get("environmentGroups", [{}])[0].get("hostnames", [])
    APIGEE_HOST = hostnames[-1] if hostnames else f"{APIGEE_ORG}-{APIGEE_ENV}.apigee.net"
except Exception as e:
    print(f"Notice retrieving hostname via Apigee API: {e}")
    APIGEE_HOST = os.environ.get("APIGEE_HOST", f"{APIGEE_ORG}-{APIGEE_ENV}.apigee.net")

os.environ["APIGEE_HOST"] = APIGEE_HOST
print(f"APIGEE_HOST: {APIGEE_HOST}")
endpoint_url = f"https://{APIGEE_HOST}/v1/chat/completions"

payload_gemini = {
    "model": "gemini-3.6-flash",
    "messages": [
        {"role": "user", "content": "Explain quantum computing in one sentence."}
    ]
}

print(f"Sending Gemini request to {endpoint_url}...")
try:
    response = requests.post(endpoint_url, headers=headers, json=payload_gemini, timeout=30)
    print(f"Status Code: {response.status_code}")
    print("Response:", json.dumps(response.json(), indent=2))
except Exception as e:
    print("Execution note:", e)


In [ ]:
# @title Test 2: OpenAI Target (GPT-4o)
import os
import json
import requests
import subprocess

APIGEE_HOST = os.getenv("APIGEE_HOST", f"{APIGEE_ORG}-{APIGEE_ENV}.apigee.net")
endpoint_url = f"https://{APIGEE_HOST}/v1/chat/completions"

# Retrieve GCP access token if headers not already set
if "headers" not in globals() or not headers.get("Authorization"):
    try:
        token = subprocess.run(
            ["gcloud", "auth", "application-default", "print-access-token"],
            capture_output=True, text=True, check=True
        ).stdout.strip()
    except Exception:
        token = subprocess.run(
            ["gcloud", "auth", "print-access-token"],
            capture_output=True, text=True
        ).stdout.strip()
    headers = {
        "Content-Type": "application/json",
        "Authorization": f"Bearer {token}"
    }

payload_openai = {
    "model": "gpt-4o",
    "messages": [
        {"role": "user", "content": "What is the capital of France?"}
    ]
}

print(f"Sending OpenAI request to {endpoint_url}...")
try:
    response = requests.post(endpoint_url, headers=headers, json=payload_openai, timeout=30)
    print(f"Status Code: {response.status_code}")
    print("Response:", json.dumps(response.json(), indent=2))
except Exception as e:
    print("Execution note:", e)


In [ ]:
# @title Test 3: Anthropic Target (Claude 3.5 Sonnet)
import os
import json
import requests
import subprocess

APIGEE_HOST = os.getenv("APIGEE_HOST", f"{APIGEE_ORG}-{APIGEE_ENV}.apigee.net")
endpoint_url = f"https://{APIGEE_HOST}/v1/chat/completions"

# Retrieve GCP access token if headers not already set
if "headers" not in globals() or not headers.get("Authorization"):
    try:
        token = subprocess.run(
            ["gcloud", "auth", "application-default", "print-access-token"],
            capture_output=True, text=True, check=True
        ).stdout.strip()
    except Exception:
        token = subprocess.run(
            ["gcloud", "auth", "print-access-token"],
            capture_output=True, text=True
        ).stdout.strip()
    headers = {
        "Content-Type": "application/json",
        "Authorization": f"Bearer {token}"
    }

payload_claude = {
    "model": "claude-3-5-sonnet-20240620",
    "messages": [
        {"role": "user", "content": "List 3 key benefits of an enterprise API gateway."}
    ],
    "max_tokens": 150
}

print(f"Sending Anthropic request to {endpoint_url}...")
try:
    response = requests.post(endpoint_url, headers=headers, json=payload_claude, timeout=30)
    print(f"Status Code: {response.status_code}")
    print("Response:", json.dumps(response.json(), indent=2))
except Exception as e:
    print("Execution note:", e)
